## Identity and Position 

In [1]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer

ALPHABET = "ء أ ؤ إ ئ ا ب ت ة ث ج ح خ د ذ ر ز س ش ص ض ط ظ ع غ ف ق ك ل م ن ه و ى ي"
MAX_LENGTH = 10


def indexes(my_list, desired_element):
    return [i + 1 for i, element in enumerate(my_list)
            if element == desired_element]


def input_id_pos(input_word, max_length=MAX_LENGTH):
    if len(input_word) > max_length:
        raise ValueError(
            f"'{input_word}' has {len(input_word)} letters; "
            f"maximum is {max_length}."
        )

    # 1) Identity: 35 dimensions
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts([ALPHABET])

    sequences = tokenizer.texts_to_sequences(list(input_word))
    flat = [value for sequence in sequences for value in sequence]

    identity = [
        '1' if index in flat else '0'
        for index in range(1, len(tokenizer.word_index) + 1)
    ]

    # 2) Position: 12 bits per letter, up to 12 letters
    
    unique_letters = sorted(set(input_word))
    positions = []

    for character in unique_letters:
        for position in indexes(list(input_word), character):
            value = 1 << (max_length - position)
            bits = f'{value:0{max_length}b}'
            positions.extend(bits)

    target_size = max_length * max_length
    positions.extend(['0'] * (target_size - len(positions)))

    return identity, positions


def save_encodings(word_list, filename, max_length=MAX_LENGTH):
    id_rows = []
    pos_rows = []

    for word in word_list:
        identity, position = input_id_pos(word, max_length)
        id_rows.append(identity)
        pos_rows.append(position)

    df_word = pd.DataFrame({'word': word_list})
    df_id = pd.DataFrame(id_rows)
    df_pos = pd.DataFrame(pos_rows)

    df_id.columns = range(1, df_id.shape[1] + 1)       # 1–35
    df_pos.columns = range(1, df_pos.shape[1] + 1)    

    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df_word.to_excel(writer, sheet_name='word', index=True)
        df_id.to_excel(writer, sheet_name='identity', index=True)
        df_pos.to_excel(writer, sheet_name='position', index=True)

In [2]:
main_doc = pd.read_excel('Large_Scale_PONG/Perea_Condition/data/words_with_transpositions_and_substitutions.xlsx', sheet_name='filtered')
words= main_doc['word'].to_list()
tl_words=main_doc['transposed_word'].to_list()
dl_words= main_doc['substitution_word'].to_list()

In [ ]:
save_encodings(words,  'Large_Scale_PONG/Perea_Condition/data/words_idpos.xlsx')
save_encodings(tl_words,  'Large_Scale_PONG/Perea_Condition/data/TL_words_idpos.xlsx')
save_encodings(dl_words,  'Large_Scale_PONG/Perea_Condition/data/DL_words_idpos.xlsx')